In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, VBox, HBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# TOEPLITZ + HANKEL COVARIANCE DECOMPOSITION
# ============================================================

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="font-family:Arial, sans-serif; font-size:16px; line-height:1.40; width:1050px;">
<div style="font-size:22px; font-weight:bold; color:#335f33; margin-bottom:8px;">Covariance Matrix of a First-Order Random System</div>
<div style="margin-bottom:4px;">Consider x[n] = αx[n−1] + w[n], with x[−1] = 0 and white-noise input of variance σ².</div>
<div style="margin-bottom:4px;">The finite-time covariance matrix can be decomposed into a Toeplitz stationary component and a Hankel transient component.</div>
<div style="margin-bottom:4px;">For |α| &lt; 1, the transient contribution becomes progressively less important with increasing time.</div>
<div><b>This notebook:</b> visualizes the exact covariance matrix and its Toeplitz–Hankel decomposition.</div>
</div>
""")

# ============================================================
# CONTROLS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='165px')

alpha_slider = FloatSlider(min=0.0, max=0.95, step=0.05, value=0.75, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)
sigma_slider = FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)
N_slider = IntSlider(min=5, max=25, step=2, value=13, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

alpha_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.75</div>')
sigma_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.0</div>')
N_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">13</div>')

def update_alpha(change):
    alpha_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{alpha_slider.value:.2f}</div>'

def update_sigma(change):
    sigma_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{sigma_slider.value:.1f}</div>'

def update_N(change):
    N_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{N_slider.value}</div>'

alpha_slider.observe(update_alpha, names='value')
sigma_slider.observe(update_sigma, names='value')
N_slider.observe(update_N, names='value')

alpha_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Coefficient α:</div>')
sigma_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Noise std σ:</div>')
N_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Matrix size:</div>')

controls_grid = GridBox(children=[alpha_label, alpha_slider, alpha_value, sigma_label, sigma_slider, sigma_value, N_label, N_slider, N_value], layout=Layout(width='760px', grid_template_columns='115px 165px 55px 115px 165px 55px', grid_template_rows='34px 34px', grid_gap='5px 8px', align_items='center', overflow='hidden'))

controls_card = VBox([HTML('<div style="font-family:Arial; font-size:17px; font-weight:bold; color:#335f33; margin-bottom:5px;">Covariance Parameters</div>'), controls_grid], layout=Layout(width='790px', padding='10px 14px', border='1px solid #bed0be', margin='10px 0px 8px 0px', overflow='hidden'))

result_html = HTML()

# ============================================================
# MAIN FUNCTION
# ============================================================

def plot_covariance_decomposition(alpha=0.75, sigma=1.0, N=13):

    indices = np.arange(N)

    A = np.zeros((N, N))

    for i in range(N):
        for j in range(i + 1):
            A[i, j] = alpha ** (i - j)

    Cww = sigma**2 * np.eye(N)

    C_exact = A @ Cww @ A.T

    if abs(alpha) < 1.0:
        stationary_variance = sigma**2 / (1.0 - alpha**2)
    else:
        stationary_variance = np.inf

    C_toeplitz = np.zeros((N, N))
    C_hankel = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            C_toeplitz[i, j] = stationary_variance * alpha ** abs(i - j)
            C_hankel[i, j] = stationary_variance * alpha ** (i + j + 2)

    C_reconstructed = C_toeplitz - C_hankel

    C_exact_norm = C_exact / stationary_variance
    C_toeplitz_norm = C_toeplitz / stationary_variance
    C_hankel_norm = C_hankel / stationary_variance

    variance_exact = np.diag(C_exact)
    variance_stationary = stationary_variance * np.ones(N)

    fig = plt.figure(figsize=(10.5, 7.2))

    gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.30)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])

    # --------------------------------------------------------
    # EXACT COVARIANCE
    # --------------------------------------------------------

    im1 = ax1.imshow(C_exact_norm, vmin=0.0, vmax=1.0, origin='upper', aspect='equal')

    ax1.set_title('Exact Normalized Covariance', fontsize=12, pad=8)
    ax1.set_xlabel('j')
    ax1.set_ylabel('i')

    fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

    # --------------------------------------------------------
    # TOEPLITZ COMPONENT
    # --------------------------------------------------------

    im2 = ax2.imshow(C_toeplitz_norm, vmin=0.0, vmax=1.0, origin='upper', aspect='equal')

    ax2.set_title('Toeplitz Stationary Component', fontsize=12, pad=8)
    ax2.set_xlabel('j')
    ax2.set_ylabel('i')

    fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    # --------------------------------------------------------
    # HANKEL COMPONENT
    # --------------------------------------------------------

    im3 = ax3.imshow(C_hankel_norm, vmin=0.0, vmax=1.0, origin='upper', aspect='equal')

    ax3.set_title('Hankel Transient Component', fontsize=12, pad=8)
    ax3.set_xlabel('j')
    ax3.set_ylabel('i')

    fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

    # --------------------------------------------------------
    # VARIANCE CONVERGENCE
    # --------------------------------------------------------

    ax4.plot(indices, variance_exact, linewidth=2.0, label='Actual variance')
    ax4.plot(indices, variance_stationary, linestyle='--', linewidth=2.0, label='Stationary variance')

    ax4.set_xlim(0, N - 1)
    ax4.set_ylim(0, 1.10 * stationary_variance)
    ax4.set_xlabel('Time index n', fontsize=11)
    ax4.set_ylabel('Variance', fontsize=11)
    ax4.set_title('Convergence to the Stationary Variance', fontsize=12, pad=8)
    ax4.grid(True, linestyle=':', alpha=0.5)
    ax4.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), fontsize=8)

    fig.subplots_adjust(left=0.07, right=0.96, top=0.92, bottom=0.13)

    plt.show()
    plt.close(fig)

    reconstruction_error = np.max(np.abs(C_exact - C_reconstructed))
    hankel_ratio = np.linalg.norm(C_hankel, 'fro') / np.linalg.norm(C_toeplitz, 'fro')

    result_html.value = f"""
    <div style="font-family:Arial; font-size:15px; line-height:1.42; width:930px; padding:10px 14px; border:1px solid #d7c38d; background:#fffbed; box-sizing:border-box;">
    <b>Stationary variance:</b> σ²/(1−α²) = {stationary_variance:.5f}
    &nbsp;&nbsp;&nbsp;
    <b>Maximum decomposition error:</b> {reconstruction_error:.3e}
    <br>
    <b>Relative Hankel energy:</b> ||H||F / ||T||F = {hankel_ratio:.5f}
    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(plot_covariance_decomposition, {'alpha': alpha_slider, 'sigma': sigma_slider, 'N': N_slider})

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="font-family:Arial; font-size:15px; line-height:1.42; width:1050px; padding:11px 15px; border:1px solid #c8dfce; background:#f8fcf9; box-sizing:border-box; margin-top:6px;">
<div style="font-size:18px; font-weight:bold; color:#335f33; margin-bottom:6px;">Interpretation of the Results</div>
<div style="margin-bottom:4px;">The Toeplitz matrix represents the stationary covariance that would remain after the transient has disappeared.</div>
<div style="margin-bottom:4px;">The Hankel matrix represents the finite-time transient caused by the zero initial condition x[−1] = 0.</div>
<div>For |α| &lt; 1, the output variance progressively approaches σ²/(1−α²) as the transient contribution decays.</div>
</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox([documentation, controls_card, output, result_html, interpretation], layout=Layout(width='1050px', overflow='hidden'))

display(main_layout)